In [5]:
import argparse
import logging
import os
import sys
import pandas as pd
from enum import StrEnum
from typing import Dict, Any

from experiments.evaluation.eval_utils import load_data, create_path_if_not_exists
from experiments.evaluation.vis_utils import save_plt, plot_boxplot, plot_multi_boxplot
from utils import setup_logging

logger = logging.getLogger(__name__)


class PlotType(StrEnum):
    LAYER_BOXPLOT = "layer_boxplot"
    LEVEL_BOXPLOT = "level_boxplot"
    MEAN_LAYER_BARPLOT = "mean_layer_barplot"
    MEAN_LEVEL_BARPLOT = "mean_level_barplot"

    def get_graph_type(self):
        parts = self.value.spit("_")
        return parts[-1]

class Metric(StrEnum):
    CONCEPT_SCORE = "concept"
    FINAL_SCORE = "final"
    FLUENCY_SCORE = "fluency"

    def get_metric_df_name(self):
        mapping = {
            Metric.CONCEPT_SCORE: "Concept Score",
            Metric.FINAL_SCORE: "Final Score",
            Metric.FLUENCY_SCORE: "Fluency Score"
        }
        return mapping[self]

class PlotMode(StrEnum):
    SINGLE_PLOT = "single_plot"
    MULTI_PLOT = "multi_plot"


MIN_SCORE = 0.0
MAX_SCORE = 2.0
BUFFER = 0.05
BAR_LABEL_FONTSIZE = 8
LAYER_COLOR_PALETTE = {
    0: "#6a3d9a",  # Purple
    1: "#fb9a99",  # Light Red
    2: "#33a02c",  # Dark Green
    3: "#ff7f00",  # Orange
    4: "#8fd7d7",  # Light Teal
    6: "#fdb462",  # Light Orange
    8: "#00b0be", # med blue
    9: "#b2df8a",  # Soft Green
    11: "#a6cee3",  # Light Blue
    12: "#b15928",  # Brown
    16: "#bdd373",  # light green
    25: "#ffffb3",  # Yellow
}

MODEL_NAME_MAP = {
    "gemma-2-2b": "Gemma-2-2B",
    "gpt2-small": "GPT-2 Small",
}



def transform_data_to_df(data: Dict[str, Any], include_fluency: bool) -> pd.DataFrame:
    """
    Transforms the hierarchical dictionary data into a flat Pandas DataFrame suitable for plotting.
    """
    records = []

    # Data is expected to be Dict[layer_str, List[entry_dict]]
    for model_name, layer_data in data.items():
        for layer_key, entries in layer_data.items():
            try:
                layer = int(layer_key)
            except ValueError:
                logger.warning(f"Skipping key '{layer_key}' as it is not a valid integer layer.")
                continue

            for entry in entries:
                level = entry.get('level')

                # Map the metric names to cleaner labels for the plot
                metrics = {
                    "Concept Score": entry.get('max_mean_concept_score'),
                    "Final Score": entry.get('max_mean_final_score')
                }

                if include_fluency:
                    metrics["Fluency Score"] = entry.get('max_mean_fluency_score')

                # Create a row for each metric
                for metric_name, score in metrics.items():
                    if score is not None:
                        records.append({
                            "Model": model_name,
                            "Layer": layer,
                            "Level": level,
                            "Metric": metric_name,
                            "Score": score
                        })

    if not records:
        logger.error("No valid records found after processing. Check input data format.")
        sys.exit(1)

    return pd.DataFrame(records)

def plot_multiple_layers_in_single_plot(df: pd.DataFrame, plot_func, plot_func_args: dict, output_dir: str, x: str, additional_plot_info: str = ""):
    logger.info("Plotting all layers in one plot per metric...")
    metrics = df['Metric'].unique()
    for metric in metrics:
        metric_df = df[df['Metric'] == metric]
        model_names = metric_df['Model'].unique()
        df_map = {}
        for model in model_names:
            plt_title = MODEL_NAME_MAP.get(model, model)
            df_map[plt_title] = metric_df[metric_df['Model'] == model]
        fig = plot_func(
            df_map=df_map,
            x=x,
            y="Score",
            hue="Layer",
            ylim=(MIN_SCORE - BUFFER, MAX_SCORE + BUFFER),
            **plot_func_args
        )

        output_path = os.path.join(output_dir, f"{additional_plot_info}_{metric.lower().replace(' ', '_')}_{x.lower()}_combined.png")
        save_plt(fig, f"Metric {metric}", output_path, logger)


def plot_layer_distribution(df: pd.DataFrame, output_dir: str, mode: PlotMode = PlotMode.MULTI_PLOT):
    """
    Generates and saves a boxplot for a specific layer across all given levels.
    """
    if mode == PlotMode.MULTI_PLOT:
        for model in df['Model'].unique():
            model_df = df[df['Model'] == model]
            if model_df.empty:
                logger.warning(f"No data found for Model {model}. Skipping plot.")
                continue
            for layer_id in model_df['Layer'].unique():
                logger.info(f"Processing layer {layer_id}...")
                layer_df = df[df['Layer'] == layer_id]
                if layer_df.empty:
                    logger.warning(f"No data found for Layer {layer_id}. Skipping plot.")
                    return

                fig = plot_boxplot(layer_df, f"Score Distribution - Layer {layer_id}", "Level", "Score", "Metric",
                                   (MIN_SCORE - BUFFER, MAX_SCORE + BUFFER))

                # Construct output filename
                folder_path = os.path.join(output_dir, model)
                create_path_if_not_exists(folder_path, is_file=False)
                output_path = os.path.join(folder_path, f"score_distribution_layer_{layer_id}.png")

                save_plt(fig, f"score distribution plot for Layer {layer_id}", output_path, logger)
    else:
        plot_multiple_layers_in_single_plot(
            df=df,
            plot_func=plot_multi_boxplot,
            plot_func_args={
                "color_palette": LAYER_COLOR_PALETTE,
            },
            output_dir=output_dir,
            x="Level",
            additional_plot_info="layer_score_distribution"
        )



In [6]:
def create_plots(output_dir: str, data_by_model: Dict[str, str], include_fluency: bool, metrics: list[str] | None, plot_mode: PlotMode):
    logger.info("Starting visualization...")

    data = {}
    for model_name, model_data in data_by_model.items():
        data[model_name] = load_data(model_data["input_file"])

    # create output directory if it doesn't exist
    create_path_if_not_exists(output_dir, is_file=False)

    df = transform_data_to_df(data, include_fluency)

    if metrics is not None:
        # Filter the dataframe if metric is given
        metric_filter = [Metric(m).get_metric_df_name() for m in metrics]
        df = df[df['Metric'].isin(metric_filter)]

    # filter relevant layers by model
    for model, model_data in data_by_model.items():
        relevant_layers = model_data.get("layers")
        if relevant_layers is not None:
            # Remove rows where the model matches and the layer is not in the relevant layers list
            df = df[~((df['Model'] == model) & (~df['Layer'].isin(relevant_layers)))]

    plot_layer_distribution(df, output_dir, mode=plot_mode)


In [8]:
data_by_model = {
    "gemma-2-2b": {
        "input_file": "experiments/results/k800/gemma-2-2b/results_in.json",
        "layers": [0,1,2,3,6,12]
    },
    "gpt2-small": {
        "input_file": "experiments/results/k800/gpt2-small/results_in.json",
        "layers": [0,1,2,3,6]
    },
    # "Llama-3.1-8B": {
    #     "input_file": "experiments/results/k400/meta-llama/Llama-3.1-8B/results_in.json",
    #     "layers": [0,1,2,3,4,8,16]
    # },
}

output_dir = "experiments/artifacts/plots/k800/multiplot"
setup_logging()

create_plots(output_dir, data_by_model, include_fluency=False, metrics=["final"], plot_mode=PlotMode.SINGLE_PLOT)

[2026-03-03 22:29:37][__main__][INFO] - Starting visualization...
[2026-03-03 22:29:37][experiments.evaluation.eval_utils][INFO] - Successfully loaded data from experiments/results/k800/gemma-2-2b/results_in.json
[2026-03-03 22:29:37][experiments.evaluation.eval_utils][INFO] - Successfully loaded data from experiments/results/k800/gpt2-small/results_in.json
[2026-03-03 22:29:37][__main__][INFO] - Plotting all layers in one plot per metric...
[2026-03-03 22:29:37][matplotlib.category][INFO] - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2026-03-03 22:29:37][matplotlib.category][INFO] - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
[2026-03-03 22:29:37][matplotlib.category][INFO] - Using categorical uni